In [1]:
# Standard library
import os
import re
import json
import subprocess
import shutil
from pathlib import Path
from pprint import pprint

# Third-party libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import faiss
import torch
from elasticsearch import Elasticsearch, helpers
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter / display utilities
from IPython.display import Markdown, display


c:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
def preprocess(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"http\S+|www\.\S+", "<URL>", s)
    s = re.sub(r"\S+@\S+", "<EMAIL>", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = ''.join(ch for ch in s if ord(ch) >= 32)
    s = re.sub(r"['\"]", "", s)
    return s.strip()

In [4]:
# Base project folders
BASE_DIR = Path.cwd()
IR_DIR   = BASE_DIR / "IR2025"
EMBED_DIR = BASE_DIR / "Embeddings"

IR_DIR.mkdir(exist_ok=True)
EMBED_DIR.mkdir(exist_ok=True)

# Core dataset paths
DOCS_CSV        = IR_DIR / "documents.csv"
DOCS_JSONL      = IR_DIR / "documents.jsonl"
QUERIES_CSV     = IR_DIR / "queries.csv"

# Embeddings
DOC_EMBED_FILE   = EMBED_DIR / "doc_embeddings.npy"
QUERY_EMBED_FILE = EMBED_DIR / "query_embeddings.npy"

# Relevance files (CSV → TXT for trec_eval)
QRELS_CSV = IR_DIR / "qrels.csv"
QRELS_TXT = IR_DIR / "qrels.txt"

# trec_eval binary
TREC_EVAL_BIN = IR_DIR / "trec_eval" / "trec_eval.exe"


In [5]:
# hyperparams
DOC_BATCH = 64
QUERY_BATCH = 32
FAISS_USE_COSINE = True
PRF_ENABLED = False              # αν θέλεις PRF ενεργό
PRF_TOP_M = 5                   # top-m docs for feedback
PRF_ALPHA = 0.7                 # weight for original query
PRF_BETA = 0.3                  # weight for feedback mean
REQUERY_CANDIDATES = 200        # initial candidate size if using candidate-limited PRF (not required)
RESULT_KS = (20, 30, 50)


In [6]:
if not DOCS_CSV.exists():
    raise SystemExit(f"documents.csv not found: {DOCS_CSV}")
if not QUERIES_CSV.exists():
    raise SystemExit(f"queries.csv not found: {QUERIES_CSV}")

df_docs = pd.read_csv(DOCS_CSV)
df_queries = pd.read_csv(QUERIES_CSV)

In [7]:
# Apply preprocess and ensure ID are strings
df_docs = df_docs.dropna(subset=["Text"]).copy()
df_docs["Text"] = df_docs["Text"].astype(str).map(preprocess)
df_docs["ID"] = df_docs["ID"].astype(str).str.strip()

df_queries = df_queries.dropna(subset=["Text"]).copy()
df_queries["Text"] = df_queries["Text"].astype(str).map(preprocess)
df_queries["ID"] = df_queries["ID"].astype(str).str.strip()

print(f"Loaded {len(df_docs)} documents; {len(df_queries)} queries.")



Loaded 18316 documents; 10 queries.


In [8]:
model = SentenceTransformer("all-mpnet-base-v2",device=device)

In [9]:
# Convert to JSONL
records_written = 0
df_docs["ID"] = df_docs["ID"].astype(str).str.strip()
with open(DOCS_JSONL, "w", encoding="utf-8") as f:
    for _, row in df_docs.iterrows():
        record = {
            "id": str(row["ID"]).strip(),
            "text": row["Text"]
        }
        json_line = json.dumps(record, ensure_ascii=False)
        f.write(json_line + "\n")
        records_written += 1

print(f"Converted {records_written} rows → JSONL format")
print(f"Output saved at: {DOCS_JSONL.resolve()}")

Converted 18316 rows → JSONL format
Output saved at: C:\Users\swkra\OneDrive - aueb.gr\Uni\7th-semester\SystemsRetr\ergasia\IR2025\documents.jsonl
